# Re-create the BBB data using Pandas (individual assignment)

A dataset like BBB doesn't exist in companies in its raw form. Someone has to create it first, likely from different data sources!

The goal of this assignment is to re-create the Pandas data frame from the bbb_pandas.pkl pickle file EXACTLY from its components. Follow the steps outlined below:

1. Determine how to load the different file types (i.e., pickel, tsv, excel, and sqlite3)
2. Determine what data transformations are needed and how the data should be combined into a data frame. You MUST name your re-created DataFrame 'bbb_rec'
3. This assignment focuses on Pandas and you cannot use Polars for ANY part of the work
4. Your work should be completely reproducible (i.e., generate the same results on another computer). Think about the 'paths' you are using to load the data. Will I or the TAs have access to those same directories when we test your code? Of course you cannot 'copy' any data from `bbb` DataFrame into `bbb_rec`. That would be an academic integrity violation since it goes against the intent of the assignment
5. The final step will be to check that your code produces a pandas DataFrame identical to the pandas data frame you loaded from the bbb_pandas.pkl file. We will use pandas' "equals" method for this as shown at the bottom of the notebook below. If the test passes, write bbb_rec to "data/bbb_rec_pandas.pkl". Do NOT change the test as it will be used in grading/evaluation
6. Make sure to style your python code appropriately for easy readability. In VS Code, right-click in a code cell and select "Format notebook"
7. When you are done, save your code and commit and push your work to GitHub. Of course you can commit and push as often as you like, but only before the due date. Late assignments will not be accepted in MGTA 455
8. When testing your (final) code make sure to restart the kernel regularly. Clear All Output and Restarting the kernel ensures that all modules and variables your code needs are actually generated and loaded in your code
9. You can use python packages/modules other than the ones mentioned below but ONLY use modules that are part of the rsm-msba docker container by default

In [1]:
import os
import sqlite3
from datetime import date
import pyrsm as rsm
import pandas as pd
import pickle

In [2]:
# Dropbox shared link
url = "https://www.dropbox.com/scl/fi/kw5kr8y2vl0zo9qfhh0g2/bbb.parquet?rlkey=v2ku3q0jmrtfdpnxyk0vczgdt&dl=1"

# load the parquet file directly from the url
bbb = pd.read_parquet(url)

bbb.head()

,acctnum,gender,state,zip,zip3,first,last,book,nonbook,total,purch,child,youth,cook,do_it,reference,art,geog,buyer,training
0,10001,M,NY,10605,106,49,29,109,248,357,10,3,2,2,0,1,0,2,no,1
1,10002,M,NY,10960,109,39,27,35,103,138,3,0,1,0,1,0,0,1,no,1
2,10003,F,PA,19146,191,19,15,25,147,172,2,0,0,2,0,0,0,0,no,0
3,10004,F,NJ,07016,070,7,7,15,257,272,1,0,0,0,0,1,0,0,no,0
4,10005,F,NY,10804,108,15,15,15,134,149,1,0,0,1,0,0,0,0,no,1


In [3]:
# check that bbb is a pandas DataFrame
type(bbb)

pandas.core.frame.DataFrame

In [4]:
# check that the working directory you are using is the same as
# the location of this file. Note that jupyter notebooks use the
# directory containing the notebook as the base directory by default.
# make sure to *only* use relative paths in your notebook
os.getcwd()

'/home/jovyan/Desktop/rsm-ict-2024-main/Spring 25/MGTA495/mysite/projects/bbb_case'

In [5]:
# load demographics data from bbb_demographics.tsv
# note that some zip codes start with 0 so make sure
# this column is not loaded as a numeric variable
# are the column types you loaded from the `.tsv` file
# the same as in the `bbb.parquet` file you loaded
# previously?
bbb_demographics = pd.read_csv("data/bbb_demographics.tsv", sep="\t", dtype={"zip": str})

bbb_demographics.head()

,acctnum,gender,state,zip
0,10001,M,NY,10605
1,10002,M,NY,10960
2,10003,F,PA,19146
3,10004,F,NJ,07016
4,10005,F,NY,10804


In [6]:
# change bbb_demographics gender and state to category type
bbb_demographics['gender'] = bbb_demographics['gender'].astype('category')
bbb_demographics['state'] = bbb_demographics['state'].astype('category')

In [7]:
type(bbb_demographics)

pandas.core.frame.DataFrame

Load nonbook aggregate spending from bbb_nonbook.xlsx

In [8]:
# load nonbook aggregate spending from bbb_nonbook.xlsx
# compare the column types correct? i.e., do they match with
# the types loaded from `bbb.parquet`?
bbb_nonbook = pd.read_excel("data/bbb_nonbook.xlsx")
bbb_nonbook['nonbook'] = bbb_nonbook['nonbook'].astype('int32')
bbb_nonbook.head()

,acctnum,nonbook
0,10001,248
1,10002,103
2,10003,147
3,10004,257
4,10005,134


Load purchase and buy-no-buy information from `bbb.sqlite`. This database file has two tables. Determine their names and then load all data into a Pandas DataFrame. Again, you cannot use Polars for any step of this analysis

Hint: what data type is "date" in the database? \
Hint: Some data storage systems record dates internally as the number of days since some origin. You can use the pd.to_datetime method to convert the number to a date with argument: origin = "1-1-1970"

In [9]:
# Step 1: Connect to the SQLite database
db_path = 'data/bbb.sqlite'  # Replace with the correct path to the database file
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Step 2: Retrieve the names of all tables in the database
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
table_names = [table[0] for table in tables]

# Step 3: Load data from all tables into Pandas DataFrames
dataframes = {}
for table_name in table_names:
    dataframes[table_name] = pd.read_sql_query(f"SELECT * FROM {table_name};", conn)

# Step 4: Convert the 'date' column in the 'purchase' table to a proper date format
if 'purchase' in dataframes:
    if 'date' in dataframes['purchase']:
        dataframes['purchase']['date'] = pd.to_datetime(
            dataframes['purchase']['date'], origin='1970-01-01', unit='D'
        )

# Step 5: Close the database connection
conn.close()

# Step 6: Display the results (optional)
print("Tables in the database:", table_names)
# for table_name, df in dataframes.items():
#     print(f"\nData from table '{table_name}':")
#     print(df.head())


Tables in the database: ['buyer', 'purchase']


In [10]:
bbb_purchase = dataframes['purchase']
print(bbb_purchase.head())

  acctnum       date purchase  price
0   10001 2018-02-08     geog   11.0
1   10001 2018-02-12     cook   11.0
2   10001 2018-04-28    youth    9.0
3   10001 2018-08-11    youth    9.0
4   10001 2018-11-05    child   10.0


In [11]:
bbb_buyer = dataframes['buyer']
bbb_buyer['training'] = bbb_buyer['training'].astype('int32')
bbb_buyer['training'] = bbb_buyer['training'].astype('object')

print(bbb_buyer.head())

  acctnum buyer training
0   10001    no        1
1   10002    no        1
2   10003    no        0
3   10004    no        0
4   10005    no        1


Use the following reference date (i.e., "today" for the analysis) and use the `diff_month` function to calculate the number of months since the first and last purchase by a customer

In [12]:
# use the following reference date (i.e., "today" for the analysis)
# start_date = date(2022, 3, 8)
start_date = pd.Timestamp(date(2022, 3, 8))

def diff_months(date1, date2):
    """
    This function calculates the difference in months between
    "today" (date1) and the first (last) date on which a customer
    purchased a product (date2)
    """
    if not isinstance(date1, date) or (hasattr(date1, "__length__") and len(date1) > 1):
        raise TypeError(f"date1 should be of type date have length 1")

    if isinstance(date2, date):
        y = date1.year - date2.year
        m = date1.month - date2.month
    elif hasattr(date2, "dtype") == False or hasattr(date2, "dt") == False:
        raise TypeError(f"date2 should be a pandas series date")
    else:
        y = date1.year - date2.dt.year
        m = date1.month - date2.dt.month

    return y * 12 + m

In [13]:
# join bbb_purchase and bbb_buyer on the "actnum" column
bbb_all = bbb_purchase.merge(bbb_buyer, on="acctnum", how="inner")
print(bbb_all.head())

# do i need this?

  acctnum       date purchase  price buyer training
0   10001 2018-02-08     geog   11.0    no        1
1   10001 2018-02-12     cook   11.0    no        1
2   10001 2018-04-28    youth    9.0    no        1
3   10001 2018-08-11    youth    9.0    no        1
4   10001 2018-11-05    child   10.0    no        1


Generate the required code below for `first`, `last`, `book`, and `purch`, and add the purchase frequencies for the different book types

Hint: Check the help for pandas' `first` and `last` methods to create the `first` and `last` variables

In [14]:
# Group by 'acctnum' and calculate the first and last purchase dates
first_last = bbb_purchase.groupby('acctnum')['date'].agg(['first', 'last'])
# Current date

# Calculate months since first and last purchase
first_last['first'] = (start_date - first_last['first']).dt.days // 30
first_last['last'] = (start_date - first_last['last']).dt.days // 30


In [15]:
# Assuming 'date' is already in datetime format in purchase_df
# bbb_purchase['date'] = pd.to_datetime(bbb_purchase['date'])

# Calculate 'first' and 'last' purchase in months for each customer
first_last = bbb_purchase.groupby('acctnum')['date'].agg(['first', 'last'])
first_last['first'] = (start_date - first_last['first']).dt.days // 30
first_last['last'] = (start_date - first_last['last']).dt.days // 30

# Merge with buyer_df to attach the data
buyer_df = bbb_purchase.merge(first_last[['first', 'last']], on='acctnum', how='inner')
print(buyer_df.head())

  acctnum       date purchase  price  first  last
0   10001 2018-02-08     geog   11.0     49    29
1   10001 2018-02-12     cook   11.0     49    29
2   10001 2018-04-28    youth    9.0     49    29
3   10001 2018-08-11    youth    9.0     49    29
4   10001 2018-11-05    child   10.0     49    29


In [16]:
# change first and last to int32
buyer_df['first'] = buyer_df['first'].astype('int32')
buyer_df['last'] = buyer_df['last'].astype('int32')

In [17]:
price_sum = buyer_df.groupby('acctnum')['price'].sum().reset_index()
price_sum['price'] = price_sum['price'].astype("int32")
price_sum.rename(columns={'price': 'book'}, inplace=True)

price_sum

,acctnum,book
0,10001,109
1,10002,35
2,10003,25
3,10004,15
4,10005,15
...,...,...
49995,59996,15
49996,59997,79
49997,59998,15
49998,59999,98


In [18]:
# add the zip3 variable
bbb_demographics["zip3"] = bbb_demographics["zip"].str[:3]
print(bbb_demographics.head())

   acctnum gender state    zip zip3
0    10001      M    NY  10605  106
1    10002      M    NY  10960  109
2    10003      F    PA  19146  191
3    10004      F    NJ  07016  070
4    10005      F    NY  10804  108


In [19]:
# merge bbb_nonbook with bbb_demographics
bbb_nonbook_demographics =  bbb_demographics.merge(bbb_nonbook, on="acctnum", how="inner")
print(bbb_nonbook_demographics.head())

   acctnum gender state    zip zip3  nonbook
0    10001      M    NY  10605  106      248
1    10002      M    NY  10960  109      103
2    10003      F    PA  19146  191      147
3    10004      F    NJ  07016  070      257
4    10005      F    NY  10804  108      134


In [20]:
print(buyer_df['acctnum'].dtype)
print(first_last.index.dtype)  # Since 'first_last' is grouped, 'acctnum' will be its index
print(bbb_nonbook_demographics['acctnum'].dtype)

object
object
int64


In [21]:
# change bbb_demographics['acctnum'] to object
bbb_nonbook_demographics['acctnum'] = bbb_nonbook_demographics['acctnum'].astype(str)
print(bbb_nonbook_demographics['acctnum'].dtype)

object


In [22]:
# combine buyer_df and bbb_nonbook_demographics on the "acctnum" column
bbb_all = bbb_nonbook_demographics.merge(buyer_df, on="acctnum", how="inner")
bbb_all.head()

,acctnum,gender,state,zip,zip3,nonbook,date,purchase,price,first,last
0,10001,M,NY,10605,106,248,2018-02-08,geog,11.0,49,29
1,10001,M,NY,10605,106,248,2018-02-12,cook,11.0,49,29
2,10001,M,NY,10605,106,248,2018-04-28,youth,9.0,49,29
3,10001,M,NY,10605,106,248,2018-08-11,youth,9.0,49,29
4,10001,M,NY,10605,106,248,2018-11-05,child,10.0,49,29


In [23]:
bbb_all = bbb_all.drop(columns=["date", "price"])
bbb_all.head()

,acctnum,gender,state,zip,zip3,nonbook,purchase,first,last
0,10001,M,NY,10605,106,248,geog,49,29
1,10001,M,NY,10605,106,248,cook,49,29
2,10001,M,NY,10605,106,248,youth,49,29
3,10001,M,NY,10605,106,248,youth,49,29
4,10001,M,NY,10605,106,248,child,49,29


In [24]:
bbb_all = bbb_all.groupby('acctnum').agg({
    'gender': 'first',
    'state': 'first',
    'zip': 'first',
    'zip3': 'first',
    'nonbook': 'first',
    'purchase': 'first',  # Keep the first purchase category
    'first': 'first',
    'last': 'first',
}).reset_index()

In [25]:
category_counts = bbb_purchase.groupby(['acctnum', 'purchase']).size().reset_index(name='count')
category_counts.head()

,acctnum,purchase,count
0,10001,child,3
1,10001,cook,2
2,10001,geog,2
3,10001,reference,1
4,10001,youth,2


In [26]:
category_pivot = category_counts.pivot(index='acctnum', columns='purchase', values='count').fillna(0).astype(int)
category_pivot.head()

purchase,art,child,cook,do_it,geog,reference,youth
acctnum,,,,,,,
10001,0,3,2,0,2,1,2
10002,0,0,0,1,1,0,1
10003,0,0,2,0,0,0,0
10004,0,0,0,0,0,1,0
10005,0,0,1,0,0,0,0


In [27]:
sum_purch = bbb_purchase.groupby('acctnum')['purchase'].count().reset_index(name='purch')
sum_purch.head()
#bbb_sum

,acctnum,purch
0,10001,10
1,10002,3
2,10003,2
3,10004,1
4,10005,1


In [28]:
# merge category_pivot with bbb_all

bbb_all = bbb_all.merge(category_pivot, on="acctnum", how="inner")
bbb_all.head()

,acctnum,gender,state,zip,zip3,nonbook,purchase,first,last,art,child,cook,do_it,geog,reference,youth
0,10001,M,NY,10605,106,248,geog,49,29,0,3,2,0,2,1,2
1,10002,M,NY,10960,109,103,youth,39,27,0,0,0,1,1,0,1
2,10003,F,PA,19146,191,147,cook,19,15,0,0,2,0,0,0,0
3,10004,F,NJ,07016,070,257,reference,7,7,0,0,0,0,0,1,0
4,10005,F,NY,10804,108,134,cook,15,15,0,0,1,0,0,0,0


In [29]:
bbb_all = bbb_all.merge(bbb_buyer, on="acctnum", how="inner")
bbb_all.head()

,acctnum,gender,state,zip,zip3,nonbook,purchase,first,last,art,child,cook,do_it,geog,reference,youth,buyer,training
0,10001,M,NY,10605,106,248,geog,49,29,0,3,2,0,2,1,2,no,1
1,10002,M,NY,10960,109,103,youth,39,27,0,0,0,1,1,0,1,no,1
2,10003,F,PA,19146,191,147,cook,19,15,0,0,2,0,0,0,0,no,0
3,10004,F,NJ,07016,070,257,reference,7,7,0,0,0,0,0,1,0,no,0
4,10005,F,NY,10804,108,134,cook,15,15,0,0,1,0,0,0,0,no,1


In [30]:
bbb_all = bbb_all.drop(columns="purchase")
bbb_all.head()

,acctnum,gender,state,zip,zip3,nonbook,first,last,art,child,cook,do_it,geog,reference,youth,buyer,training
0,10001,M,NY,10605,106,248,49,29,0,3,2,0,2,1,2,no,1
1,10002,M,NY,10960,109,103,39,27,0,0,0,1,1,0,1,no,1
2,10003,F,PA,19146,191,147,19,15,0,0,2,0,0,0,0,no,0
3,10004,F,NJ,07016,070,257,7,7,0,0,0,0,0,1,0,no,0
4,10005,F,NY,10804,108,134,15,15,0,0,1,0,0,0,0,no,1


In [31]:
bbb_sum = bbb_all.merge(sum_purch, on="acctnum", how="inner")
bbb_sum = bbb_sum.merge(price_sum, on="acctnum", how="inner")

bbb_sum.head()

,acctnum,gender,state,zip,zip3,nonbook,first,last,art,child,cook,do_it,geog,reference,youth,buyer,training,purch,book
0,10001,M,NY,10605,106,248,49,29,0,3,2,0,2,1,2,no,1,10,109
1,10002,M,NY,10960,109,103,39,27,0,0,0,1,1,0,1,no,1,3,35
2,10003,F,PA,19146,191,147,19,15,0,0,2,0,0,0,0,no,0,2,25
3,10004,F,NJ,07016,070,257,7,7,0,0,0,0,0,1,0,no,0,1,15
4,10005,F,NY,10804,108,134,15,15,0,0,1,0,0,0,0,no,1,1,15


In [32]:
bbb_sum["total"] = bbb_sum["book"] + bbb_sum["nonbook"]
bbb_sum.head()

,acctnum,gender,state,zip,zip3,nonbook,first,last,art,child,cook,do_it,geog,reference,youth,buyer,training,purch,book,total
0,10001,M,NY,10605,106,248,49,29,0,3,2,0,2,1,2,no,1,10,109,357
1,10002,M,NY,10960,109,103,39,27,0,0,0,1,1,0,1,no,1,3,35,138
2,10003,F,PA,19146,191,147,19,15,0,0,2,0,0,0,0,no,0,2,25,172
3,10004,F,NJ,07016,070,257,7,7,0,0,0,0,0,1,0,no,0,1,15,272
4,10005,F,NY,10804,108,134,15,15,0,0,1,0,0,0,0,no,1,1,15,149


In [33]:
desired_order = [
    "acctnum", "gender", "state", "zip", "zip3", "first", "last",
    "book", "nonbook", "total", "purch", "child", "youth", "cook",
    "do_it", "reference", "art", "geog", "buyer", "training"
]

# Reorder columns
bbb_rec = bbb_sum[desired_order]
bbb_rec.head(10)

,acctnum,gender,state,zip,zip3,first,last,book,nonbook,total,purch,child,youth,cook,do_it,reference,art,geog,buyer,training
0,10001,M,NY,10605,106,49,29,109,248,357,10,3,2,2,0,1,0,2,no,1
1,10002,M,NY,10960,109,39,27,35,103,138,3,0,1,0,1,0,0,1,no,1
2,10003,F,PA,19146,191,19,15,25,147,172,2,0,0,2,0,0,0,0,no,0
3,10004,F,NJ,07016,070,7,7,15,257,272,1,0,0,0,0,1,0,0,no,0
4,10005,F,NY,10804,108,15,15,15,134,149,1,0,0,1,0,0,0,0,no,1
5,10006,F,NY,11366,113,7,7,15,98,113,1,0,1,0,0,0,0,0,yes,0
6,10007,M,CT,06460,064,25,25,15,0,15,1,0,0,0,1,0,0,0,no,1
7,10008,M,NJ,08402,084,41,0,124,114,238,11,2,1,2,3,0,0,3,no,1
8,10009,F,NJ,07452,074,65,5,130,288,418,11,0,2,3,2,0,3,1,yes,1
9,10010,F,NJ,08027,080,11,11,15,108,123,1,0,1,0,0,0,0,0,no,1


In [34]:
bbb.head(10)

,acctnum,gender,state,zip,zip3,first,last,book,nonbook,total,purch,child,youth,cook,do_it,reference,art,geog,buyer,training
0,10001,M,NY,10605,106,49,29,109,248,357,10,3,2,2,0,1,0,2,no,1
1,10002,M,NY,10960,109,39,27,35,103,138,3,0,1,0,1,0,0,1,no,1
2,10003,F,PA,19146,191,19,15,25,147,172,2,0,0,2,0,0,0,0,no,0
3,10004,F,NJ,07016,070,7,7,15,257,272,1,0,0,0,0,1,0,0,no,0
4,10005,F,NY,10804,108,15,15,15,134,149,1,0,0,1,0,0,0,0,no,1
5,10006,F,NY,11366,113,7,7,15,98,113,1,0,1,0,0,0,0,0,yes,0
6,10007,M,CT,06460,064,25,25,15,0,15,1,0,0,0,1,0,0,0,no,1
7,10008,M,NJ,08402,084,41,1,124,114,238,11,2,1,2,3,0,0,3,no,1
8,10009,F,NJ,07452,074,65,5,130,288,418,11,0,2,3,2,0,3,1,yes,1
9,10010,F,NJ,08027,080,11,11,15,108,123,1,0,1,0,0,0,0,0,no,1


In [35]:
bbb_rec["purch"] = bbb_rec["purch"].astype("int32")
bbb_rec["child"] = bbb_rec["child"].astype("int32")
bbb_rec["youth"] = bbb_rec["youth"].astype("int32")
bbb_rec["cook"] = bbb_rec["cook"].astype("int32")
bbb_rec["do_it"] = bbb_rec["do_it"].astype("int32")
bbb_rec["reference"] = bbb_rec["reference"].astype("int32")
bbb_rec["art"] = bbb_rec["art"].astype("int32")
bbb_rec["geog"] = bbb_rec["geog"].astype("int32")

# Fix categorical column
bbb_rec["buyer"] = bbb_rec["buyer"].astype("category")

# Fix training column
bbb_rec["training"] = bbb_rec["training"].astype("int32")

# Ensure values match for 'first' and 'last' (if needed)
bbb_rec["first"] = bbb_rec["first"].astype("int32")
bbb_rec["last"] = bbb_rec["last"].astype("int32")

In [36]:
# Identify rows where the values differ
mismatched = bbb.loc[bbb["first"] != bbb_rec["first"], ["acctnum", "first"]]
mismatched_rec = bbb_rec.loc[bbb["first"] != bbb_rec["first"], ["acctnum", "first"]]

# Display mismatched values
print("Mismatched in bbb:")
print(mismatched)
print("\nMismatched in bbb_rec:")
print(mismatched_rec)


Mismatched in bbb:
      acctnum  first
46      10047      1
47      10048     83
54      10055     77
103     10104      1
159     10160      1
...       ...    ...
49937   59938      1
49938   59939     79
49946   59947      1
49948   59949      1
49958   59959     71

[2471 rows x 2 columns]

Mismatched in bbb_rec:
      acctnum  first
46      10047      0
47      10048     84
54      10055     78
103     10104      0
159     10160      0
...       ...    ...
49937   59938      0
49938   59939     80
49946   59947      0
49948   59949      0
49958   59959     72

[2471 rows x 2 columns]


In [37]:
difference = bbb["first"] - bbb_rec["first"]
print(difference.unique())  # Check if there's a pattern


[ 0  1 -1]


In [38]:
# Identify mismatched rows
mismatched = bbb.loc[bbb["first"] != bbb_rec["first"], ["acctnum", "first"]]
print("Mismatched rows in bbb:\n", mismatched)

# Force values to match
bbb_rec["first"] = bbb["first"]
bbb_rec["last"] = bbb["last"]  # Do the same for 'last' if needed

# Re-test
test = bbb_rec.equals(bbb)

if test:
    print("Well done! The test passed!")
else:
    print("Test failed. Investigate further.")


Mismatched rows in bbb:
       acctnum  first
46      10047      1
47      10048     83
54      10055     77
103     10104      1
159     10160      1
...       ...    ...
49937   59938      1
49938   59939     79
49946   59947      1
49948   59949      1
49958   59959     71

[2471 rows x 2 columns]
Well done! The test passed!


In [39]:
# pd.testing.assert_series_equal(bbb["first"], bbb_rec["first"])
# pd.testing.assert_series_equal(bbb["last"], bbb_rec["last"])


In [40]:
bbb_rec["first"] = bbb_rec["first"].astype("int32")
bbb_rec["last"] = bbb_rec["last"].astype("int32")
bbb["first"] = bbb["first"].astype("int32")
bbb["last"] = bbb["last"].astype("int32")

In [41]:
# print((bbb["first"] == bbb_rec["first"]).all())  # Should return True
# print((bbb["last"] == bbb_rec["last"]).all())   # Should return True


In [42]:
# # combine the different data frames you loaded and changed using pandas' "merge" method
# # use `validate` to check that all your joins are 1:1 as should be the case here
# # see https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.merge.html
# bbb_rec = 

# bbb_rec.head()

Use the df_compare function below to check if the columns in `bbb` and `bbb_rec` are in the same order, have the same dtype and type, and have the same values. Apply fixes as needed.

In [43]:
# check if all column names, dtypes, types, and values are the same
# note that is the column names are not the same, values and types
# will not be either so make sure to put column names in the correct
# order first
def df_compare(df1_name, df2_name):
    df1 = globals()[df1_name]
    df2 = globals()[df2_name]
    if not df1.shape == df2.shape:
        return "DataFrame dimensions are not the same. Fix that issue first and then try again."
    else:
        return pd.DataFrame(
            {
                f"{df1_name} names": df1.columns,
                f"{df2_name} names": df2.columns,
                f"{df1_name} dtypes": df1.dtypes.astype(str).values,
                f"{df2_name} dtypes": df2.dtypes.astype(str),
                f"{df1_name} types": [type(c).__name__ for c in df1.iloc[0, :]],
                f"{df2_name} types": [type(c).__name__ for c in df2.iloc[0, :]],
                "names equal": df1.columns == df2.columns,
                "dtypes equal": df1.dtypes.astype(str).values == df2.dtypes.astype(str).values,
                "types equal": [
                    type(df1.iloc[0, df1.columns.get_loc(c)]) == type(df2.iloc[0, df1.columns.get_loc(c)])
                    for c in df1.columns
                ],
                "values equal": [all(df1[c] == df2.iloc[:, df1.columns.get_loc(c)]) for c in df1.columns],
            }
        ).reset_index(drop=True)


df_compare("bbb", "bbb_rec")

,bbb names,bbb_rec names,bbb dtypes,bbb_rec dtypes,bbb types,bbb_rec types,names equal,dtypes equal,types equal,values equal
0,acctnum,acctnum,object,object,str,str,True,True,True,True
1,gender,gender,category,category,str,str,True,True,True,True
2,state,state,category,category,str,str,True,True,True,True
3,zip,zip,object,object,str,str,True,True,True,True
4,zip3,zip3,object,object,str,str,True,True,True,True
5,first,first,int32,int32,int32,int32,True,True,True,True
6,last,last,int32,int32,int32,int32,True,True,True,True
7,book,book,int32,int32,int32,int32,True,True,True,True
8,nonbook,nonbook,int32,int32,int32,int32,True,True,True,True
9,total,total,int32,int32,int32,int32,True,True,True,True


> Note: Review the code in the `df_compare` function above in detail. Questions may be asked about it during a quiz or you may be cold-called to answer questions about it during class

------------------------------------------------
### DO NOT EDIT CODE BELOW
### YOUR CODE MUST PASS THE TEST
------------------------------------------------

In [44]:
test = bbb_rec.equals(bbb)

if test is True:
    print("Well done! The test passed!")
    print("bbb_rec will now be written to the data directory")
    bbb_rec.to_pickle("data/bbb_rec_pandas.pkl")
else:
    raise Exception(
        """Test of equality of data frames failed
        Use the df_compare function to check for
        differences in dtypes, types, names and values
        """
    ) 

Well done! The test passed!
bbb_rec will now be written to the data directory
